In [ ]:
import numpy as np

# Lab 08: Variational Inference

In this session, we explore how to use Variational Inference in two different problems: to infer the parameter of a simple decision model with fixed-form VI, and to infer the parameters of a univariate normal distribution.

## Part 1: Fixed-form VI for a simple decision model

We consider a problem where a user has to make a binary decision based on some characteristics $x$. The decision, denoted by $y$, is either to accept or to reject $x$. We consider a unique user with a fixed decision policy (no evolution over time). We observe several decisions of the user, combined into a dataset $\mathcal{D} = \lbrace (x_1, y_1), \dotsc, (x_N, y_N) \rbrace$.

### Probabilistic model

We model this process as follows:
* Likelihood: $p(y_n = 1 \, | \, x_n, z) = \frac{1}{1 + \exp(- z x_n)}$
* Prior: $p(z \, | \, \mu, \sigma^2) = \mathcal{N}(z ; \mu, \sigma^2)$

In this model, $z$ is a real-valued latent parameter which indicates the tendency of the user to accept or reject the decisions. We can see that a high $z$ corresponds to a tendency to accept, while a low $z$ corresponds to a tendency to reject. Note that this corresponds to a Bernoulli distribution, the parameter of which depends on $x_n$. 

#### Question 1

Give an expression of $\log p(y_n \, | \, x_n, z)$ for any $y_n$ and implement it.

In [ ]:
def log_likelihood(y, x, z):
    ### Your code here
    return 0

#### Question 2

Express the ELBO $\text{L}(\mu, \sigma^2, \psi \, | \, \mathbf{X}, \mathbf{y})$ for a generic variational distribution $q_{\psi}$.

<hr>

We will now use variational inference (VI) to infer the posterior over the latent variable $z$. Note that the parameters $\mu, \sigma^2$ are the parameters of the prior and supposed to be known. We will not consider the inference of these parameters in this session. 



### Reminder: Laplace approximation
 
We first consider that the variational distribution is a normal distribution:
$$
q_{m, s^2}(z) = \mathcal{N}(z; m, s^2)
$$
This choice of a variational distribution is called Laplace approximation. 

The goal is now to use a gradient-based method to optimize the ELBO with respect to $m$ and $s^2$. A difficulty here is that the gradient cannot be computed in closed-form, so we need to use an approximation. For this, we use Monte-Carlo sampling. 


### Monte-Carlo Sampling

The base idea behind Monte-Carlo sampling is to estimate an expectation $\mathbb{E}[f(X)]$ by first sampling $J$ i.i.d. values from random variable $X$, from which we have:
$$
\mathbb{E}[f(X)] \simeq \frac{1}{J} \sum_{j=1}^J f(x_j)
$$
With $f(x) = x$, this corresponds to the estimation of the mean of a sample. 

We illustrate this idea with a simple example. Let $X_1$ and $X_2$ be two random variables following a normal distribution of respective parameters $(\mu_1, \sigma_1)^2$ and $(\mu_2, \sigma_2)^2$. It can be verified (and this is a good exercise) that $X_1 + X_2$ follows a normal distribution of mean $\mu_1 + \mu_2$ and of variance $\sigma_1^2 + \sigma_2^2$.


#### Question 3

Implement a Monte-Carlo sampling of $X_1 + X_2$ (where the parameters of $X_1$ and $X_2$ can be chosen arbitrarily) and verify that it indeed converges to $\mathbb{E}[X_1 + X_2] = \mu_1 + \mu_2$.

In [ ]:
def MC_sampling_sum_gaussians(mu1, sigma1, mu2, sigma2, n_samples):
    ### Your code here
    return 0

# Choice of the normal distributions:
mu1 = 2
sigma1 = 0.5
mu2 = 3
sigma2 = 0.2

n_samples = 100

print("MC estimation:", MC_sampling_sum_gaussians(mu1, sigma1, mu2, sigma2, n_samples))
print("Real value:", mu1 + mu2)

### Reparametrization trick

To use a gradient descent, we need to compute the gradient of the ELBO with respect to $\psi = (m, s^2)$. We would like to use Monte-Carlo sampling for this, but this is not possible, $\nabla_\psi \mathbb{E}_{Z \sim q_\psi}[f(Z)] \neq \mathbb{E}_{Z \sim q_\psi}[\nabla_\psi f(Z)]$. We solve this problem by using the reparametrization trick, which consists in changing the variable in the expectation to make its distribution independ from $\psi$. 

#### Question 4

Show that:
$$
\mathbb{E}_{Z \sim \mathcal{N}(m, s^2)}[f(Z)] = \mathbb{E}_{Z^\prime \sim \mathcal{N}(0, 1)}[f(\mu + s Z^\prime)]
$$


#### Question 5 (optional)

Using the result of Question 4 and noticing that it is now possible to permute the gradient and the expectation, implement a Monte-Carlo sampling of the gradient of the ELBO. You can either compute the gradient manually or using automatic tools. 

## Part 2: Free-form VI

We assume a dataset $\mathcal{D} = \lbrace x_1, \dotsc, x_N \rbrace$ and consider the following model:
* Likelihood: Normal distribution of mean $\mu$ and precision $\lambda = 1 / \sigma^2$:
$$
p(\mathcal{D} \, | \, \theta) = \prod_{n=1}^N \mathcal{N}(x_n \, | \, \mu, \lambda^{-1})
$$
* Prior: Conjugate prior for the normal distribution: 
$$
p(\mu, \lambda) = \text{Gamma}(\lambda \, | \, a_0, b_0) \, \mathcal{N}(\mu \, | \, \mu_0, (\kappa_0 \lambda)^{-1})
$$

We use the mean field approximation: 
$$
q(\mu, \lambda) = q_{\mu}(\mu | \psi_\mu) q_{\lambda}(\lambda | \psi_\lambda)
$$

#### Question 6

Compute the log probability for joint distribution $\log p(\mu, \lambda, \mathcal{D})$.

#### Question 7

Show that $q_{\mu}(\mu | \psi_\mu)$ is a normal distribution. Express its mean and variance. 

#### Question 8 

Show that $q_{\lambda}(\lambda | \psi_\lambda)$ is a Gamma distribution. Express its parameters.

#### Question 9

Use the two obtained distributions to compute the missing parameters (the expectation terms in the parameters you have). 